In [ ]:
%reload_ext autoreload
%autoreload 2

import os
import sys
from pathlib import Path

project_root = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src" / "global_settings.py").is_file()
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
os.chdir(project_root)
from src.global_settings import settings


In [ ]:
import math

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm import tqdm

from src.activations import ActivationDataset
from src.sae import SparseAutoencoder, SAELoss
from src.utils.seed import seed_worker

# SparseAutoencoder, SAELoss 테스트

In [ ]:
DATA_DIR = "./data/post_mlp_residual_stream/layer_11"

BASE_SEED = 42
BATCH_SIZE = 128

g = torch.Generator()
g.manual_seed(BASE_SEED)

dataset = ActivationDataset(data_dir=DATA_DIR, shuffle=True)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, num_workers=0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
model = SparseAutoencoder(
    architecture="topk",
    topk=64,
    input_dim=768,
    feature_dim=1536,
    apply_auxiliary_loss=True,
    k_aux=512,
    alpha_aux=1/32,
    dead_threshold=500,
)

model.to(device)
model.train()

model.topk, model.encoder.weight.shape, model.decoder.weight.shape

### single-batch feed-forwarding test

In [ ]:
batch = next(iter(dataloader))[0]
inputs = batch.to(device)

features, reconstructed, loss_dict = model(inputs, return_loss_components=True)

print("inputs:", inputs.shape)
print("features:", features.shape)
print("reconstructed:", reconstructed.shape)
if loss_dict:
    for key, value in loss_dict.items():
        if isinstance(value, torch.Tensor):
            print(F"{key}: {value.item():.4f}")
        else:
            print(F"{key}: {value}")
    
    total_loss = loss_dict["reconstruction_loss"] + model.alpha_aux * loss_dict["auxiliary_loss"]
    print(f"total_loss: {total_loss.item():.4f}")

### full-batch feed-forwarding test

In [ ]:
num_total_batches = (len(dataset) + BATCH_SIZE - 1) // BATCH_SIZE

In [ ]:
len(dataset), num_total_batches

In [ ]:
total_metrics = {
    "reconstruction_loss": 0.0,
    "auxiliary_loss": 0.0,
    "feature_density": 0.0,
    "dead_features": 0,
}
num_batches = 0

with torch.no_grad():
    for i, batch in enumerate(tqdm(dataloader, total=num_total_batches)):
        inputs = batch.to(device)
        features, reconstructed, loss_dict = model(inputs, return_loss_components=True)

        if loss_dict:
            total_metrics["reconstruction_loss"] += loss_dict["reconstruction_loss"].item()
            total_metrics["auxiliary_loss"] += loss_dict["auxiliary_loss"].item()
            total_metrics["feature_density"] += loss_dict["feature_density"].item()
            total_metrics["dead_features"] = loss_dict["dead_features"]
        
        num_batches += 1

print("--- Full-batch Feedforwarding Results ---")
if num_batches > 0:
    avg_metrics = {key: value / num_batches for key, value in total_metrics.items() if key != "dead_features"}
    avg_metrics["dead_features"] = total_metrics["dead_features"]
    print("Average Metrics over all batches:")
    for key, value in avg_metrics.items():
        if isinstance(value, float):
            print(f"  - {key}: {value:.6f}")
        else:
            print(f"  - {key}: {value}")

## SparseAutoencoder unit test

In [ ]:
INPUT_DIM = 768
FEATURE_DIM = 1536
TOPK = 64

SEQ_LEN = 2048

### (1) _initialize_weights

In [ ]:
sample_data = next(iter(dataset))

(1.1) `apply_unit_norm` 확인

In [ ]:
sae_after = SparseAutoencoder(
    architecture="topk",
    topk=TOPK,
    input_dim=INPUT_DIM,
    feature_dim=FEATURE_DIM,
    apply_transpose_init=True,
    apply_unit_norm=True,
)
W_enc_after = sae_after.encoder.weight.detach()
W_dec_after = sae_after.decoder.weight.detach()
norm_dec_after = torch.norm(W_dec_after, p=2, dim=0)

norm_dec_after.shape, norm_dec_after

(1.2) `apply_tranpose_init` 확인

In [ ]:
normalized_transposed_encoder_w = F.normalize(W_enc_after.t(), p=2, dim=0)
is_tranposed_and_normalized = torch.allclose(W_dec_after, normalized_transposed_encoder_w)

is_tranposed_and_normalized

In [ ]:
normalized_transposed_encoder_w[:, :5], W_dec_after[:, :5]

In [ ]:
sae = SparseAutoencoder(
    architecture="topk",
    topk=TOPK,
    input_dim=INPUT_DIM,
    feature_dim=FEATURE_DIM,
    apply_transpose_init=True,
    apply_unit_norm=False,
)
W_enc = sae.encoder.weight.detach()
W_dec = sae.decoder.weight.detach()

In [ ]:
W_enc.shape, W_dec.shape

In [ ]:
W_enc @ W_dec

In [ ]:
torch.allclose(W_enc, W_dec.t())

### (2) `forward()` 함수 내 모든 서브모듈 확인

In [ ]:
sae = SparseAutoencoder(
    architecture="topk",
    input_dim=INPUT_DIM,
    feature_dim=FEATURE_DIM,
    topk=TOPK,
    apply_preprocessing=True,
    apply_auxiliary_loss=True,
).train()

In [ ]:
data_iterator = iter(dataset)
sample_batch = torch.stack([next(data_iterator) for _ in range(BATCH_SIZE)])

- (1) 'processed_input = self._preprocess_input(x)'에서 `processed`가 예상과 일치하는 데이터가 출력되는지 확인;

In [ ]:
print("(1) Testing _preprocess_input...")

processed_input = sae._preprocess_input(sample_batch)
input_vector_norm = torch.norm(processed_input[0, 0, :], p=2)
expected_norm = math.sqrt(INPUT_DIM)
print(f"  - `processed_input` shape: {processed_input.shape}")
print(f"  - `processed_input` mean (should be ~0): {processed_input.mean():.4f}")
print(f"  - A single vector's L2 norm (should be ~sqrt({INPUT_DIM})={expected_norm:.2f}): {input_vector_norm:.2f}")

assert processed_input.shape == sample_batch.shape
assert torch.allclose(input_vector_norm, torch.tensor(expected_norm), atol=1e-3)

print("  - ✅ OK")

- (2) 'encoded = self.encoder(processed_input)'에서 출력된 encoded의 shape가 올바른지 확인;

In [ ]:
print("(2) Testing encoder...")

processed_input = sae._preprocess_input(sample_batch)
encoded = sae.encoder(processed_input)
print(f"  - `encoded` shape: {encoded.shape}")

assert encoded.shape == (BATCH_SIZE, SEQ_LEN, FEATURE_DIM)
print("  - ✅ OK")

- (3, 중요!) 'features = self._apply_activation(encoded)'에서 각 구조별 activation 함수가 올바르게 작동하는지 확인:
    - F.relu(encoded)
    - `_topk_activation` (특히, 모델 인자 중 `TOPK`와 같은 활성화 tensor가 계산되는지 확인)
    - `_gated_activation`
    - `_prolu_activation`
    - 샘플 데이터에 대한 features 텐서의 값 및 shape을 출력

In [ ]:
print("(3) Testing _apply_activation (topk, relu, gated, prolu)...")

for arch in ["topk", "relu","prolu"]:
    print(f"  - Architecture: {arch}")
    sae = SparseAutoencoder(
        architecture=arch,
        input_dim=INPUT_DIM,
        feature_dim=FEATURE_DIM,
        topk=TOPK,
        apply_auxiliary_loss=False,
    ).train()

    processed_input = sae._preprocess_input(sample_batch)
    encoded = sae.encoder(processed_input)
    features = sae._apply_activation(encoded)
    
    print(f"    - `features` shape: {features.shape}")
    print(f"    - `features` sample values (first 5): {features.view(-1)[:5]}")

    assert features.shape == encoded.shape

    if arch == "relu":
        assert (features >= 0).all()
        print("  - ReLU check: All feature values are non-negative. ✅ OK")
    elif arch == "topk":
        non_zero_counts = torch.count_nonzero(features, dim=-1)
        assert (non_zero_counts <= TOPK).all()
        print(f"  - Top-K check: Non-zero features per vector <= {TOPK}. ✅ OK")
    
print("  - ✅ OK")

In [ ]:
sae = SparseAutoencoder(
    architecture="topk",
    input_dim=INPUT_DIM,
    feature_dim=FEATURE_DIM,
    topk=TOPK,
    apply_auxiliary_loss=True,
).train()

In [ ]:
processed_input = sae._preprocess_input(sample_batch)
encoded = sae.encoder(processed_input)
features = sae._apply_activation(encoded)

In [ ]:
encoded_flat = encoded.view(-1, FEATURE_DIM)
topk_values, topk_indices = torch.topk(encoded_flat, TOPK, dim=-1)

In [ ]:
encoded_flat.shape

In [ ]:
topk_values, topk_indices, topk_values.shape

### 해당 batch(128개 샘플)에서 TOP-64에 포함된 특징(index in 0 ~ 1536) 순위:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

indices_flat = topk_indices.view(-1)
feature_counts = torch.bincount(indices_flat, minlength=FEATURE_DIM)
counts_np = feature_counts.cpu().numpy()

top_n_indices = np.argsort(counts_np)[::-1][:TOPK]
top_n_counts = counts_np[top_n_indices]

plt.figure()
plt.bar(range(TOPK), top_n_counts, color='deepskyblue', edgecolor='black')
plt.title(f'Top {TOPK} Most Frequent Features', fontsize=16)
plt.xlabel('Feature Index', fontsize=12)
plt.ylabel('Number of Times Selected (Count)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

In [ ]:
top_n_counts

In [ ]:
features_flat = torch.zeros_like(encoded_flat)

features_flat

In [ ]:
feats_flat = torch.zeros_like(encoded_flat)
feats = feats_flat.scatter_(-1, topk_indices, topk_values)
feats = feats.view_as(encoded)

In [ ]:
feats.shape, feats

- (4) 'reconstruced = self.decoder(features)' 텐서 값 및 shape을 출력

In [ ]:
print("(4) Testing decoder...")

sae = SparseAutoencoder(
    architecture="topk",
    input_dim=INPUT_DIM,
    feature_dim=FEATURE_DIM,
    topk=TOPK,
    apply_auxiliary_loss=True,
).train()

processed_input = sae._preprocess_input(sample_batch)
encoded = sae.encoder(processed_input)
features = sae._apply_activation(encoded)

reconstructed = sae.decoder(features)

print(f"  - `reconstructed` shape: {reconstructed.shape}")
print(f"  - `reconstructed` sample values (first 5): {reconstructed.view(-1)[:5]}")
assert reconstructed.shape == sample_batch.shape
print("  - ✅ OK")

- (5) `apply_preprocessing`이 True일 경우 적용되어야 하는 'reconsturcted = reconstructed + self.activation_mean'의 출력 결과를 확인하고, 의도대로 정상 작동된 결과인지 확인

In [ ]:
print("(5) Testing de-centering...")
reconstructed_final = reconstructed + sae.activation_mean
print(f"  - `reconstructed_final` shape: {reconstructed_final.shape}")
print(f"  - `activation_mean` (first 5): {sae.activation_mean[:5]}")
print(f"  - `reconstructed_final` mean (should be close to activation_mean): {reconstructed_final.mean(dim=[0,1])[:5]}")
print("  - ✅ OK")

- (6) `flag_squeezed`의 값을 출력하고, features 및 reconstructed가 `squeeze(0)` 적용되었을 때 문제 없는 로직인 점을 확인

In [ ]:
sample_single = sample_batch[0, ...]

sample_single.shape

In [ ]:
print("(6) Testing squeeze logic with 2D input...")
features_2d, reconstructed_2d, _ = sae.forward(sample_single, return_loss_components=False)
print(f"  - Input shape: {sample_single.shape} (2D)")
print(f"  - Output `features` shape: {features_2d.shape} (should be 2D)")
print(f"  - Output `reconstructed` shape: {reconstructed_2d.shape} (should be 2D)")
assert features_2d.dim() == 2
assert reconstructed_2d.dim() == 2
print("  - ✅ OK")

- (7) loss_components의 모든 항목의 중간 값을 출력

In [ ]:
print("(7) Testing loss_components...")
_, _, loss_components = sae.forward(sample_batch, return_loss_components=True)
print("  - `loss_components` dict:")
for key, value in loss_components.items():
    print(f"    - {key}: {value.item() if isinstance(value, torch.Tensor) else value}")
assert "reconstruction_loss" in loss_components
assert "feature_density" in loss_components
assert "dead_features" in loss_components
assert "auxiliary_loss" in loss_components
print("  - ✅ OK")